# 02 — Embeddings and Vector Stores

**Repo:** ai-learning-notes | **Folder:** rag-and-retrieval  
**Built alongside:** local-rag-llm (github.com/RT91-data/local-rag-llm)

---

## What this notebook covers

- What an embedding actually is — vectors, dimensions, semantic space
- How transformer encoders produce embeddings internally
- Distance metrics — L2 vs cosine vs dot product, and which FAISS uses
- The L2-to-similarity conversion in local-rag-llm and why it matters
- FAISS index types — Flat, IVF, HNSW — and when to use each
- nomic-embed-text — why it was chosen and what makes it different
- The similarity threshold of 0.3 — what it means and how to tune it
- Embedding model comparison

**Pure Python only — no external dependencies.**

---
## 1. What Is an Embedding?

An embedding is a fixed-length list of numbers — a vector — that represents the **semantic meaning** of a piece of text.

- nomic-embed-text produces **768-dimensional** vectors
- OpenAI text-embedding-3-small produces **1536-dimensional** vectors
- Each number is a float between roughly -1 and 1

**The key property:** texts with similar meaning have vectors that point in similar directions in 768-dimensional space.

"Zero Ambient Authority" and "agents should not inherit full admin privileges" will produce vectors that are close together, even though they share no words.

"Zero Ambient Authority" and "the weather in Singapore" will produce vectors that are far apart.

**What the individual dimensions mean:** Nothing interpretable on their own. The model learned to distribute meaning across all 768 dimensions jointly during training. You cannot look at dimension 42 and say it represents "security". Meaning lives in the *relationships between dimensions*, not in individual ones.

In [ ]:
import math
import random

# What an embedding looks like
# Real nomic-embed-text vectors are 768-dimensional
# We show a 12-dimensional slice for readability

random.seed(42)

def mock_embed(text, dims=12):
    """NOT a real embedding model. Illustrates shape only."""
    random.seed(hash(text) % 2**31)
    vec = [random.gauss(0, 1) for _ in range(dims)]
    # Normalise to unit length (like real embedding models do)
    magnitude = math.sqrt(sum(x**2 for x in vec))
    return [x / magnitude for x in vec]

texts = [
    "Zero Ambient Authority means agents must not inherit full admin privileges.",
    "Agents should never be granted a global key or broad standing permissions.",
    "The weather in Singapore is hot and humid throughout the year.",
]

print("WHAT AN EMBEDDING LOOKS LIKE (12D slice of a 768D vector)")
print("=" * 70)
for text in texts:
    vec = mock_embed(text)
    formatted = ", ".join(f"{v:+.3f}" for v in vec)
    print(f"Text: '{text[:55]}...'")
    print(f"Vec:  [{formatted}]")
    print()

print("Each number individually is meaningless.")
print("Meaning is encoded in the RELATIVE POSITIONS of vectors in 768D space.")

---
## 2. How Transformer Encoders Produce Embeddings

This is the internal mechanism — simplified but mechanistically accurate.

**Step 1 — Tokenisation**  
Text is split into subword tokens using a vocabulary of ~50,000 entries.  
"Zero Ambient Authority" → `["Zero", "▁Ambient", "▁Authority"]`  
"SPIFFE" might become `["SP", "IFF", "E"]` if it's not in the vocabulary.

**Step 2 — Token Embeddings**  
Each token is looked up in a learned embedding table → a 768D vector per token.

**Step 3 — Transformer Layers**  
The sequence of token vectors passes through N transformer layers.  
Each layer runs **multi-head self-attention**: every token attends to every other token.  
This lets "Authority" in "Zero Ambient Authority" be influenced by "Zero" and "Ambient".

**Step 4 — Pooling**  
After the final transformer layer, you have one 768D vector per token.  
To get one vector for the whole text, pool across tokens:  
- **CLS pooling**: use the special `[CLS]` token's final vector (BERT-style)
- **Mean pooling**: average all token vectors (nomic-embed-text uses this)
- **Max pooling**: take the maximum value per dimension

nomic-embed-text uses **mean pooling with normalisation** — the average of all token vectors, then scaled to unit length.

In [ ]:
# Illustrating mean pooling — how token vectors become one document vector

def mean_pool(token_vectors):
    """Average across all token vectors, then normalise to unit length."""
    dims = len(token_vectors[0])
    pooled = [sum(tv[d] for tv in token_vectors) / len(token_vectors) for d in range(dims)]
    magnitude = math.sqrt(sum(x**2 for x in pooled))
    return [x / magnitude for x in pooled]

# Simulated token-level embeddings (4D for illustration)
# In reality these come from the final transformer layer
token_vectors = {
    "Zero":      [0.8,  0.2, -0.1,  0.3],
    "Ambient":   [0.7,  0.4,  0.0,  0.2],
    "Authority": [0.6,  0.5,  0.1,  0.1],
    "means":     [0.1,  0.1,  0.8,  0.1],
    "agents":    [0.2,  0.3,  0.6,  0.2],
}

token_list = list(token_vectors.values())
sentence_embedding = mean_pool(token_list)

print("MEAN POOLING — from token vectors to sentence vector")
print("=" * 55)
print(f"{'Token':<12} {'d0':>8} {'d1':>8} {'d2':>8} {'d3':>8}")
print("-" * 50)
for token, vec in token_vectors.items():
    print(f"{token:<12} {vec[0]:>8.3f} {vec[1]:>8.3f} {vec[2]:>8.3f} {vec[3]:>8.3f}")
print("-" * 50)
raw_mean = [sum(tv[d] for tv in token_list) / len(token_list) for d in range(4)]
print(f"{'mean':<12} {raw_mean[0]:>8.3f} {raw_mean[1]:>8.3f} {raw_mean[2]:>8.3f} {raw_mean[3]:>8.3f}")
print(f"{'normalised':<12} {sentence_embedding[0]:>8.3f} {sentence_embedding[1]:>8.3f} {sentence_embedding[2]:>8.3f} {sentence_embedding[3]:>8.3f}")
print()
magnitude = math.sqrt(sum(x**2 for x in sentence_embedding))
print(f"Magnitude after normalisation: {magnitude:.6f} (should be 1.0)")
print("Unit normalisation ensures all vectors live on the same scale.")

---
## 3. Distance Metrics — L2 vs Cosine vs Dot Product

Once you have embedding vectors, you need a way to measure how similar two vectors are.  
Three metrics are used in practice:

**L2 Distance (Euclidean)**  
Straight-line distance between two points in N-dimensional space.  
Range: 0 (identical) to ∞ (completely different).  
**FAISS uses L2 by default.**

**Cosine Similarity**  
The angle between two vectors — ignores magnitude, only considers direction.  
Range: -1 (opposite) to +1 (identical).  
Preferred when vectors are not normalised to unit length.

**Dot Product**  
If vectors are unit-normalised, dot product = cosine similarity.  
Faster to compute than cosine (no division).  
nomic-embed-text normalises its output → dot product and cosine are equivalent.

**Key insight for local-rag-llm:**  
FAISS uses L2 distance. Lower L2 = more similar.  
The code converts L2 to a 0–1 similarity score using `1 / (1 + L2_score)`.  
This is why the similarity threshold of 0.3 means something specific — it filters out chunks where L2 > 2.33.

In [ ]:
def l2_distance(a, b):
    return math.sqrt(sum((x - y)**2 for x, y in zip(a, b)))

def cosine_similarity(a, b):
    dot = sum(x * y for x, y in zip(a, b))
    norm_a = math.sqrt(sum(x**2 for x in a))
    norm_b = math.sqrt(sum(x**2 for x in b))
    return dot / (norm_a * norm_b) if norm_a and norm_b else 0

def dot_product(a, b):
    return sum(x * y for x, y in zip(a, b))

def l2_to_similarity(l2):
    """Conversion used in local-rag-llm retrieve_with_threshold()"""
    return 1 / (1 + l2)

# Unit-normalised vectors (as nomic-embed-text produces)
def normalise(v):
    mag = math.sqrt(sum(x**2 for x in v))
    return [x / mag for x in v]

# Simulated embeddings (4D)
pairs = [
    ("ZAA text",    "similar: JIT tokens",  [0.8, 0.3, 0.1, 0.2], [0.75, 0.35, 0.15, 0.25]),
    ("ZAA text",    "unrelated: weather",   [0.8, 0.3, 0.1, 0.2], [0.1, -0.5, 0.7, -0.3]),
    ("ZAA text",    "identical copy",       [0.8, 0.3, 0.1, 0.2], [0.8, 0.3, 0.1, 0.2]),
]

print(f"{'Pair':<35} {'L2':>8} {'Sim(L2)':>9} {'Cosine':>8} {'Dot':>8}")
print("-" * 75)
for label_a, label_b, vec_a, vec_b in pairs:
    a = normalise(vec_a)
    b = normalise(vec_b)
    l2 = l2_distance(a, b)
    sim = l2_to_similarity(l2)
    cos = cosine_similarity(a, b)
    dot = dot_product(a, b)
    pair_label = f"{label_a[:12]} / {label_b[:18]}"
    print(f"{pair_label:<35} {l2:>8.4f} {sim:>9.4f} {cos:>8.4f} {dot:>8.4f}")

print()
print("For unit-normalised vectors: cosine == dot product (as expected).")
print("L2-to-similarity formula: 1/(1+L2). Used in local-rag-llm threshold filter.")

In [ ]:
# Understanding the similarity threshold of 0.3
# From local-rag-llm: SIMILARITY_THRESHOLD = 0.3
# similarity = 1 / (1 + l2_score)
# threshold 0.3 means: keep chunks where 1/(1+l2) >= 0.3
# solving: 1+l2 <= 1/0.3 = 3.33, so l2 <= 2.33

print("SIMILARITY THRESHOLD = 0.3 — what it means")
print("=" * 55)
print()
print("Formula: similarity = 1 / (1 + L2_distance)")
print()
print(f"{'L2 distance':>14} {'Similarity':>12} {'Kept?':>8}")
print("-" * 40)

l2_values = [0.0, 0.2, 0.5, 0.8, 1.0, 1.5, 2.0, 2.33, 2.5, 3.0, 5.0]
for l2 in l2_values:
    sim = 1 / (1 + l2)
    kept = "✅ Yes" if sim >= 0.3 else "❌ No"
    print(f"{l2:>14.2f} {sim:>12.4f} {kept:>8}")

print()
print("Threshold 0.3 = cutoff at L2 distance > 2.33")
print("Chunks beyond that distance are discarded before reranking.")
print()
print("Tuning advice:")
print("  Lower threshold (0.2): retrieve more chunks, more noise, better recall")
print("  Higher threshold (0.4): retrieve fewer chunks, cleaner, risk missing answers")
print("  0.3 is a reasonable default — tune via RAGAS context_recall")

---
## 4. FAISS Index Types

FAISS is not a single algorithm — it is a library of different index structures.  
The choice of index type determines the tradeoff between speed, memory, and accuracy.

**IndexFlatL2 (what local-rag-llm uses)**  
- Brute-force exact search: compare query to every vector
- 100% accuracy — guaranteed to find the true nearest neighbours
- Memory: O(n × d) floats — one float per dimension per vector
- Speed: O(n × d) per query — scales linearly with index size
- **Best for:** small-to-medium indexes (up to ~100K vectors)

**IndexIVFFlat**  
- Divides vectors into `nlist` clusters using k-means at build time
- At query time: find the nearest `nprobe` clusters, then search within them
- Approximate — trades recall for speed
- **Best for:** 100K–10M vectors

**IndexHNSW (Hierarchical Navigable Small World)**  
- Graph-based index — builds a layered network of vector connections
- Very fast queries, good recall, higher memory than IVF
- **Best for:** low-latency production deployments

**For local-rag-llm:**  
IndexFlatL2 is the right choice. A 50-page PDF produces ~250 chunks. Brute-force over 250 vectors takes microseconds. The accuracy guarantee matters more than speed at this scale.

In [ ]:
# Simulating FAISS IndexFlatL2 brute-force search

def brute_force_l2_search(query_vec, index_vecs, k=3):
    """
    Exact nearest-neighbour search — what IndexFlatL2 does.
    Returns top-k (index, distance) pairs sorted by distance.
    """
    distances = []
    for i, vec in enumerate(index_vecs):
        dist = l2_distance(query_vec, vec)
        distances.append((i, dist))
    return sorted(distances, key=lambda x: x[1])[:k]


random.seed(0)

# Simulate a small FAISS index: 10 chunks, 4D embeddings
chunk_texts = [
    "Zero Ambient Authority — agents never inherit full admin privileges.",
    "JIT downscoping provides task-scoped credentials that expire immediately.",
    "The Confused Deputy problem occurs when prompt injection tricks an agent.",
    "SPIFFE IDs provide cryptographic identity to every agent.",
    "Ephemeral sandboxes reset state between each vibe loop iteration.",
    "The Vibe Diff translates generated code into plain English for approval.",
    "RAGAS measures faithfulness, answer relevancy, precision, and recall.",
    "BM25 uses term frequency and inverse document frequency for ranking.",
    "CrossEncoder rerankers read question and chunk together for accuracy.",
    "Context recall measures whether retrieved chunks contain the full answer.",
]

# Create fake 4D embeddings
def embed_4d(text):
    random.seed(hash(text) % 2**31)
    v = [random.gauss(0, 1) for _ in range(4)]
    mag = math.sqrt(sum(x**2 for x in v))
    return [x/mag for x in v]

index_vectors = [embed_4d(t) for t in chunk_texts]

query = "How do agents get assigned minimal credentials?"
query_vec = embed_4d(query)

results = brute_force_l2_search(query_vec, index_vectors, k=3)

print("FAISS IndexFlatL2 — Brute Force Search")
print("=" * 65)
print(f"Query: '{query}'")
print()
print(f"{'Rank':<6} {'L2 dist':>9} {'Sim':>7}  Chunk")
print("-" * 75)
for rank, (idx, dist) in enumerate(results, 1):
    sim = 1 / (1 + dist)
    kept = "✅" if sim >= 0.3 else "❌"
    print(f"{rank:<6} {dist:>9.4f} {sim:>7.4f} {kept} {chunk_texts[idx][:55]}...")

print()
print(f"Index size: {len(index_vectors)} vectors × 4 dims")
print(f"Real local-rag-llm: ~250 vectors × 768 dims (nomic-embed-text)")
print(f"Brute force over 250×768 floats: microseconds on CPU — no approximation needed.")

---
## 5. nomic-embed-text — Why This Model?

nomic-embed-text is the embedding model used in local-rag-llm. Key properties:

| Property | Value | Notes |
|---|---|---|
| **Dimensions** | 768 | Standard BERT-sized, good balance |
| **Max input tokens** | 8192 | Can embed very long chunks without truncation |
| **Model size** | ~274MB | Runs comfortably on CPU |
| **License** | Apache 2.0 | Free for commercial use |
| **Runs via** | Ollama | Local, no API key, no cost per call |
| **MTEB score** | ~62 (avg) | Competitive with OpenAI ada-002 |

**Why local over OpenAI embeddings?**  
- Zero cost per embedding call (important when indexing thousands of chunks)
- No data leaving your machine (critical for confidential ERP data)
- 8192 token context window vs 8191 for ada-002 — effectively the same

**The tradeoff:**  
OpenAI text-embedding-3-large (3072 dimensions) outperforms nomic-embed-text on most benchmarks.  
For local document RAG where privacy matters, nomic-embed-text is the right call.  
For a production SaaS product where accuracy is paramount, use a hosted model.

In [ ]:
# Embedding model comparison

models = [
    {
        "name": "nomic-embed-text",
        "dims": 768,
        "max_tokens": 8192,
        "size_mb": 274,
        "cost_per_1k": 0.0,
        "runs_local": True,
        "mteb_avg": 62.4,
        "notes": "Used in local-rag-llm",
    },
    {
        "name": "text-embedding-ada-002",
        "dims": 1536,
        "max_tokens": 8191,
        "size_mb": None,
        "cost_per_1k": 0.0001,
        "runs_local": False,
        "mteb_avg": 60.9,
        "notes": "OpenAI, data leaves machine",
    },
    {
        "name": "text-embedding-3-small",
        "dims": 1536,
        "max_tokens": 8191,
        "size_mb": None,
        "cost_per_1k": 0.00002,
        "runs_local": False,
        "mteb_avg": 62.3,
        "notes": "OpenAI, cheapest hosted",
    },
    {
        "name": "text-embedding-3-large",
        "dims": 3072,
        "max_tokens": 8191,
        "size_mb": None,
        "cost_per_1k": 0.00013,
        "runs_local": False,
        "mteb_avg": 64.6,
        "notes": "OpenAI, highest accuracy",
    },
    {
        "name": "BGE-large-en-v1.5",
        "dims": 1024,
        "max_tokens": 512,
        "size_mb": 1340,
        "cost_per_1k": 0.0,
        "runs_local": True,
        "mteb_avg": 63.6,
        "notes": "Local, best open-source, needs GPU",
    },
]

print(f"{'Model':<28} {'Dims':>5} {'MaxTok':>7} {'$/1K':>7} {'Local':>6} {'MTEB':>6}  Notes")
print("-" * 95)
for m in models:
    local = "✅" if m["runs_local"] else "❌"
    cost = f"{m['cost_per_1k']:.5f}" if m['cost_per_1k'] > 0 else "free"
    print(
        f"{m['name']:<28} {m['dims']:>5} {m['max_tokens']:>7} {cost:>7} "
        f"{local:>6} {m['mteb_avg']:>6.1f}  {m['notes']}"
    )

print()
print("For 250 chunks (typical 50-page PDF):")
chunks = 250
avg_tokens_per_chunk = 500
total_tokens = chunks * avg_tokens_per_chunk
print(f"  Total tokens to embed: ~{total_tokens:,}")
print(f"  nomic-embed-text cost: $0.00 (local)")
print(f"  text-embedding-3-large: ~${total_tokens/1000 * 0.00013:.4f}")
print("  Cost difference negligible at this scale — privacy is the deciding factor.")

---
## 6. How FAISS Saves and Loads the Index

FAISS supports serialising the index to disk via `write_index` and `read_index`.  
LangChain wraps this as `vectorstore.save_local(path)` and `FAISS.load_local(path, embeddings)`.

**What gets saved:**
- The FAISS index file (`index.faiss`) — the raw vectors and index structure
- The docstore file (`index.pkl`) — mapping from FAISS IDs back to LangChain Documents

**The `allow_dangerous_deserialization=True` flag:**  
The docstore is saved as a Python pickle file.  
Pickle can execute arbitrary code when loaded — this is a known security risk.  
LangChain added the flag to force developers to acknowledge this risk explicitly.  
In local-rag-llm, the index is written and read by the same app on the same machine — acceptable risk.  
In a cloud deployment, use a managed vector DB instead (Pinecone, Azure AI Search, Weaviate).

**What BM25 can't do:**  
BM25 has no `save_local` equivalent in LangChain's implementation.  
It must be rebuilt from the raw documents on every startup.  
This is why `load_chunks_from_disk()` in local-rag-llm re-reads the PDFs on each app restart.

In [ ]:
# Illustrating the index persistence pattern from local-rag-llm

INDEX_DIR = "faiss_index_storage"

# What LangChain saves to disk
index_files = {
    f"{INDEX_DIR}/index.faiss": "Binary FAISS index — the raw vectors + index structure",
    f"{INDEX_DIR}/index.pkl":   "Python pickle — LangChain docstore (chunk text + metadata)",
}

print("FILES WRITTEN BY vectorstore.save_local()")
print("=" * 65)
for filepath, description in index_files.items():
    print(f"  {filepath}")
    print(f"  → {description}")
    print()

print("LOADING PATTERN in app.py:")
print("""
vectorstore = FAISS.load_local(
    INDEX_DIR,
    embeddings,                         # same model used at index time
    allow_dangerous_deserialization=True  # acknowledges pickle risk
)
""")

print("WHY BM25 IS REBUILT ON EVERY STARTUP:")
print("""
# BM25Retriever has no save_local() method in LangChain
# Must rebuild from source documents every time
chunks = load_chunks_from_disk()  # reads PDFs from temp_uploads/
bm25_ret = BM25Retriever.from_documents(chunks)

# For a 50-page PDF → ~250 chunks → rebuild takes ~2-5 seconds
# Acceptable for single-user local deployment
# For production: use Elasticsearch or BM25S (has pickle support)
""")

---
## 7. Memory and Storage Estimates

Understanding how much memory FAISS uses is important for production sizing.

In [ ]:
# FAISS memory footprint calculation
# IndexFlatL2 stores raw float32 vectors
# Memory = n_vectors × n_dimensions × 4 bytes (float32)

DIMS = 768          # nomic-embed-text dimensions
BYTES_PER_FLOAT = 4 # float32

scenarios = [
    ("50-page PDF",           250),
    ("10 PDFs (500 pages)",  2_500),
    ("D365 FnO help (est.)", 50_000),
    ("Large doc library",    500_000),
    ("Switch to IVF needed", 1_000_000),
]

print(f"FAISS IndexFlatL2 Memory Usage ({DIMS}D, float32)")
print("=" * 60)
print(f"{'Scenario':<28} {'Chunks':>8} {'Memory':>10} {'Index type'}")
print("-" * 65)

for label, n_chunks in scenarios:
    bytes_used = n_chunks * DIMS * BYTES_PER_FLOAT
    if bytes_used < 1024**2:
        mem_str = f"{bytes_used/1024:.1f} KB"
    elif bytes_used < 1024**3:
        mem_str = f"{bytes_used/1024**2:.1f} MB"
    else:
        mem_str = f"{bytes_used/1024**3:.2f} GB"

    if n_chunks < 100_000:
        index_type = "IndexFlatL2 ✅"
    elif n_chunks < 1_000_000:
        index_type = "Consider IVF"
    else:
        index_type = "IVF or HNSW required"

    print(f"{label:<28} {n_chunks:>8,} {mem_str:>10}  {index_type}")

print()
print("local-rag-llm at typical scale: <5MB FAISS index — well within any machine's RAM.")
print("IndexFlatL2 is the right choice. No approximation needed.")

---
## 8. The Full Embedding → Storage → Retrieval Flow

Putting it all together — from raw document to retrieved chunk.

In [ ]:
# End-to-end flow illustration

print("INDEXING PHASE (runs once when PDFs are uploaded)")
print("=" * 60)
print("""
1. Load PDF with pdfplumber
   → Document objects with page_content and metadata

2. Split with RecursiveCharacterTextSplitter
   chunk_size=2000, chunk_overlap=400
   → ~250 chunks per 50-page PDF

3. Security scan each chunk (Layer 1)
   → Remove injection attempts before they enter the index

4. Embed each chunk
   OllamaEmbeddings(model='nomic-embed-text')
   → 250 × 768-dimensional float vectors

5. Build FAISS IndexFlatL2
   → Add all 250 vectors to the index

6. Save to disk
   vectorstore.save_local('faiss_index_storage')
   → index.faiss + index.pkl
""")

print("QUERY PHASE (runs per question)")
print("=" * 60)
print("""
1. Embed the query
   OllamaEmbeddings(model='nomic-embed-text')
   → 1 × 768-dimensional vector

2. FAISS similarity search (k=16)
   vectorstore.similarity_search_with_score(query, k=16)
   → 16 (chunk, L2_distance) pairs

3. Similarity threshold filter
   similarity = 1 / (1 + L2_distance)
   Keep only chunks where similarity >= 0.3
   → Typically 8-12 chunks pass

4. BM25 retrieval (k=4)
   bm25_ret.invoke(query)
   → 4 chunks by keyword frequency

5. Junk chunk filter
   Remove TOC pages, header-only chunks

6. Merge + deduplicate
   → ~12-14 unique candidates

7. CrossEncoder rerank (top_k=5)
   ms-marco-MiniLM-L-6-v2
   → 5 final chunks, ordered by true relevance

8. Send to Claude with source citations
   → Answer grounded in retrieved context
""")

---
## 9. Summary

| Concept | Key point |
|---|---|
| **Embedding** | Fixed-length vector encoding semantic meaning — 768D for nomic-embed-text |
| **Dimensions** | Individually meaningless — meaning is in relative positions across all dims |
| **Pooling** | nomic-embed-text uses mean pooling + unit normalisation |
| **L2 distance** | What FAISS uses — lower = more similar — range 0 to ∞ |
| **Cosine similarity** | For unit vectors: equivalent to dot product and to 1/(1+L2) conversion |
| **Threshold 0.3** | Drops chunks where L2 > 2.33 — filters irrelevant noise |
| **IndexFlatL2** | Brute-force exact search — correct for <100K vectors, microsecond latency |
| **IVF / HNSW** | Approximate search — use when index exceeds 100K vectors |
| **nomic-embed-text** | Local, free, 8192 token window, Apache 2.0 — right choice for private docs |
| **Pickle risk** | FAISS docstore uses pickle — acceptable local, use managed DB in production |

---
## Next Notebooks

- **03** — Sparse retrieval and BM25 (term frequency, IDF, when keywords beat vectors)
- **04** — Hybrid retrieval (EnsembleRetriever, weighting, Reciprocal Rank Fusion)
- **05** — Reranking with CrossEncoders (bi-encoder vs cross-encoder, ms-marco)